# EduVision_DV — 02. Data Cleaning

In [2]:
import pandas as pd
import numpy as np
import re
import os

RAW_DIR = os.path.join(".", "..", "data", "raw")
CLEANED_DIR = os.path.join(".", "..", "data", "cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)


def standardize_columns(df):
    df.columns = (
        df.columns.str.strip().str.lower()
        .str.replace(" ", "_").str.replace("-", "_").str.replace(":", "_")
    )
    return df


def clean_university_name(name):
    if pd.isna(name):
        return name
    name = str(name).strip().lower()
    name = re.sub(r"\([^)]*\)", " ", name)
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


COUNTRY_MAP = {
    "united states of america": "united states", "usa": "united states", "u.s.a.": "united states",
    "uk": "united kingdom", "u.k.": "united kingdom",
    "russian federation": "russia",
    "republic of korea": "south korea", "korea, republic of": "south korea",
    "iran, islamic republic of": "iran", "islamic republic of iran": "iran",
    "china (mainland)": "china", "mainland china": "china",
    "hong kong sar": "hong kong", "hong kong (china)": "hong kong",
    "macau sar": "macau", "macau (china)": "macau",
    "czech republic": "czechia",
}


def clean_country(country):
    if pd.isna(country):
        return country
    c = str(country).lower().strip()
    c = COUNTRY_MAP.get(c, c)
    return re.sub(r"\s+", " ", c).strip()


def parse_rank_band(value):
    """Both THE-family datasets switch from exact rank to text bands
    lower down ('=193', '201-250', '1401+', 'Reporter', '-'). Keep the
    raw text but derive a usable numeric field (band midpoint) plus
    flags - this is NOT the same as inventing a precise rank."""
    if pd.isna(value):
        return pd.Series({"rank_numeric": np.nan, "rank_is_banded": False})
    s = str(value).strip().lstrip("=")
    if s in ("-", "", "Reporter"):
        return pd.Series({"rank_numeric": np.nan, "rank_is_banded": True})
    if s.replace(".", "", 1).isdigit():
        return pd.Series({"rank_numeric": float(s), "rank_is_banded": False})
    s = s.replace("–", "-").replace("—", "-")
    if "-" in s:
        try:
            lo, hi = (float(x) for x in s.split("-", 1))
            return pd.Series({"rank_numeric": (lo + hi) / 2, "rank_is_banded": True})
        except ValueError:
            pass
    if s.endswith("+"):
        try:
            return pd.Series({"rank_numeric": float(s[:-1]) + 100, "rank_is_banded": True})
        except ValueError:
            pass
    return pd.Series({"rank_numeric": np.nan, "rank_is_banded": True})


# ----------------------------------------------------------------------
def clean_qs():
    print("=" * 60, "\nCLEAN: QS World University Rankings 2025\n" + "=" * 60)
    df = pd.read_csv(os.path.join(RAW_DIR, "qs_2025_raw.csv"), encoding="latin1")
    df = standardize_columns(df)

    before = len(df)
    df = df.drop_duplicates()
    df = df.dropna(subset=["institution_name"])
    df["institution_name"] = df["institution_name"].astype(str).str.strip()  # raw QS names carry trailing whitespace
    print(f"Rows: {before} -> {len(df)} (dropped exact dup / blank-name rows)")

    df["university_name_clean"] = df["institution_name"].apply(clean_university_name)
    df["country_clean"] = df["location"].apply(clean_country)

    rank_info = df["rank_2025"].apply(parse_rank_band)
    df = pd.concat([df, rank_info], axis=1)

    score_cols = [c for c in df.columns if c.endswith("_score")]
    for c in score_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")  # NaN stays NaN - never filled with 0

    dup_names = df["university_name_clean"].duplicated().sum()
    print(f"Duplicate standardized names: {dup_names}")
    print(f"Missing overall_score (exact): {df['overall_score'].isna().sum()} / {len(df)} "
          f"(expected - QS only scores its top tier)")

    out = os.path.join(CLEANED_DIR, "qs_2025_clean.csv")
    df.to_csv(out, index=False)
    print("saved ->", out)
    return df


def clean_the():
    print("\n" + "=" * 60, "\nCLEAN: Times Higher Education World University Rankings 2024\n" + "=" * 60)
    df = pd.read_csv(os.path.join(RAW_DIR, "the_2024_raw.csv"))
    df = standardize_columns(df)

    before = len(df)
    df = df.drop_duplicates()
    df = df.dropna(subset=["name"])
    df["name"] = df["name"].astype(str).str.strip()  # THE raw names carry trailing whitespace
    print(f"Rows: {before} -> {len(df)} (dropped exact dup / blank-name rows)")
    # NOTE: 'record_type' (master_account/public/private) is a THE account
    # tier flag, not a duplicate marker - every row is a distinct institution.

    df["university_name_clean"] = df["name"].apply(clean_university_name)
    df["country_clean"] = df["location"].apply(clean_country)

    rank_info = df["rank"].apply(parse_rank_band)
    df = pd.concat([df, rank_info], axis=1)

    score_cols = [c for c in df.columns if c.startswith("scores_")]
    for c in score_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["stats_number_students"] = pd.to_numeric(
        df["stats_number_students"].astype(str).str.replace(",", "", regex=False), errors="coerce"
    )
    df["stats_student_staff_ratio"] = pd.to_numeric(df["stats_student_staff_ratio"], errors="coerce")
    df["stats_pc_intl_students"] = pd.to_numeric(
        df["stats_pc_intl_students"].astype(str).str.replace("%", "", regex=False), errors="coerce"
    )

    dup_names = df["university_name_clean"].duplicated().sum()
    print(f"Duplicate standardized names: {dup_names}")
    print(f"Missing scores_overall (exact): {df['scores_overall'].isna().sum()} / {len(df)} "
          f"(expected - THE only scores its top tier)")

    out = os.path.join(CLEANED_DIR, "the_2024_clean.csv")
    df.to_csv(out, index=False)
    print("saved ->", out)
    return df


def clean_wur():
    print("\n" + "=" * 60, "\nCLEAN: World University Rankings 2023\n" + "=" * 60)
    df = pd.read_csv(os.path.join(RAW_DIR, "wur_2023_raw.csv"))
    df = standardize_columns(df)

    before = len(df)
    # Investigated first (see 01_dataset_validation_report.md): the 29
    # duplicate rows and most of the 107 duplicate names are blank
    # trailer rows at the end of the scrape (rank='-', name=NaN), not
    # real duplicate institutions - drop those specifically.
    df = df.dropna(subset=["name_of_university"])
    df = df.drop_duplicates()
    print(f"Rows: {before} -> {len(df)} (dropped blank-name trailer rows + exact dup rows)")

    remaining_dups = df[df["name_of_university"].duplicated(keep=False)].sort_values("name_of_university")
    print(f"Remaining duplicate names after dropping blanks: "
          f"{df['name_of_university'].duplicated().sum()} (kept - could be legitimately "
          f"same-named branch campuses; flagged for manual review, not auto-removed)")

    df["university_name_clean"] = df["name_of_university"].apply(clean_university_name)
    df["country_clean"] = df["location"].apply(clean_country)

    rank_info = df["university_rank"].apply(parse_rank_band)
    df = pd.concat([df, rank_info], axis=1)

    df["no_of_student"] = pd.to_numeric(
        df["no_of_student"].astype(str).str.replace(",", "", regex=False), errors="coerce"
    )
    df["no_of_student_per_staff"] = pd.to_numeric(df["no_of_student_per_staff"], errors="coerce")
    df["international_student"] = pd.to_numeric(
        df["international_student"].astype(str).str.replace("%", "", regex=False), errors="coerce"
    )
    for c in ["overall_score", "teaching_score", "research_score",
              "citations_score", "industry_income_score", "international_outlook_score"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    out = os.path.join(CLEANED_DIR, "wur_2023_clean.csv")
    df.to_csv(out, index=False)
    print("saved ->", out)
    return df


# ----------------------------------------------------------------------
# Dataset 4 — World Bank Education Statistics (country-level, different shape)
# ----------------------------------------------------------------------
WORLD_BANK_FILE = "world_bank_education_subset.csv"
WORLD_BANK_INDICATORS = {
    "SE.XPD.TOTL.GD.ZS": "govt_expenditure_pct_gdp",
    "SE.TER.ENRR": "tertiary_enrollment_ratio_pct",
    "SE.ADT.LITR.ZS": "adult_literacy_rate_pct",
    "SE.XPD.TOTL.GB.ZS": "govt_expenditure_pct_govt_spending",
    "SE.PRM.ENRR": "primary_enrollment_ratio_pct",
}


def clean_world_bank():
    print("\n" + "=" * 60, "\nCLEAN: World Bank Education Statistics\n" + "=" * 60)
    raw_path = os.path.join(RAW_DIR, WORLD_BANK_FILE)
    if not os.path.exists(raw_path):
        print(f"NOT YET PROVIDED - expected at {raw_path}. Skipping this dataset for now; "
              f"every other step in the pipeline works without it. Re-run once the file is added.")
        return None

    df = pd.read_csv(raw_path)
    before = len(df)
    df = df.drop_duplicates()

    # 1. Keep only the 5 indicators the project actually needs (Section 5) -
    #    not all ~4,000 in the raw EdStats file.
    df = df[df["Indicator Code"].isin(WORLD_BANK_INDICATORS.keys())].copy()
    print(f"Rows: {before} (full file) -> {len(df)} (filtered to {len(WORLD_BANK_INDICATORS)} selected indicators)")

    # 2. Reshape from wide (one column per year) to long
    year_cols = [c for c in df.columns if c.strip().isdigit()]
    long_df = df.melt(
        id_vars=["Country Name", "Country Code", "Indicator Code"],
        value_vars=year_cols, var_name="year", value_name="value"
    )
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    missing_before = long_df["value"].isna().mean()
    long_df = long_df.dropna(subset=["value"])  # a missing year-value is dropped, never filled with 0
    print(f"Missing values in the year x indicator grid: {missing_before:.1%} "
          f"(expected - EdStats reporting years vary widely by country; rows dropped, not zero-filled)")

    # 3. Keep the most recent available year per country+indicator
    long_df = long_df.sort_values("year").groupby(["Country Name", "Country Code", "Indicator Code"]).tail(1)

    # 4. Standardize country names with the SAME mapping used for the
    #    university datasets, so country_clean values line up in Step 3.
    long_df["country_clean"] = long_df["Country Name"].apply(clean_country)
    long_df["indicator_name"] = long_df["Indicator Code"].map(WORLD_BANK_INDICATORS)

    dup_pairs = long_df.duplicated(subset=["country_clean", "Indicator Code"]).sum()
    print(f"Duplicate (country, indicator) pairs after keeping latest year: {dup_pairs}")

    out = os.path.join(CLEANED_DIR, "world_bank_education_clean.csv")
    long_df.to_csv(out, index=False)
    print("saved ->", out)
    print(f"Countries covered: {long_df['country_clean'].nunique()}")
    return long_df


if __name__ == "__main__":
    qs = clean_qs()
    the = clean_the()
    wur = clean_wur()
    world_bank = clean_world_bank()
    print("\nFinal cleaned shapes:", qs.shape, the.shape, wur.shape,
          world_bank.shape if world_bank is not None else "(World Bank - not yet provided)")


CLEAN: QS World University Rankings 2025
Rows: 1503 -> 1503 (dropped exact dup / blank-name rows)
Duplicate standardized names: 2
Missing overall_score (exact): 903 / 1503 (expected - QS only scores its top tier)
saved -> .\..\data\cleaned\qs_2025_clean.csv

CLEAN: Times Higher Education World University Rankings 2024
Rows: 2673 -> 2673 (dropped exact dup / blank-name rows)
Duplicate standardized names: 1
Missing scores_overall (exact): 2472 / 2673 (expected - THE only scores its top tier)
saved -> .\..\data\cleaned\the_2024_clean.csv

CLEAN: World University Rankings 2023
Rows: 2341 -> 2233 (dropped blank-name trailer rows + exact dup rows)
Remaining duplicate names after dropping blanks: 0 (kept - could be legitimately same-named branch campuses; flagged for manual review, not auto-removed)
saved -> .\..\data\cleaned\wur_2023_clean.csv

CLEAN: World Bank Education Statistics
Rows: 1210 (full file) -> 1210 (filtered to 5 selected indicators)
Missing values in the year x indicator grid